# 3D-HyperGAMO

**Trạng thái:** `SOURCE_LOCKED`  
**Chế độ mặc định:** CPU · read-only · không train · không resume · không chạy test inference.  

Kiến trúc 3D-HyperGAMO và đối chiếu implementation M4. Protocol B không được gọi là tái lập nguyên trạng toàn bộ paper.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'manifest.json').is_file() and (ROOT.parent / 'manifest.json').is_file():
    ROOT = ROOT.parent
elif not (ROOT / 'manifest.json').is_file() and (ROOT / 'KLTN_clean' / 'manifest.json').is_file():
    ROOT = ROOT / 'KLTN_clean'
if not (ROOT / 'manifest.json').is_file():
    raise FileNotFoundError('Project root not found. Open the extracted project folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.audit import (verify_sources, scan, zip_listing, zip_job_contracts,
                       read_test_summary, read_test_by_seed, npz_headers,
                       source_functions, function_source, statuses, export_json)

# Optional directories containing original ZIPs and data.
import json
from src.audit import resolve_inputs
INPUT_CONFIG = resolve_inputs(ROOT)
INPUTS = INPUT_CONFIG['directories']
from src.results import (read_csv_file_or_zip, metrics_by_seed,
                         test_paired_deltas, recompute_summary,
                         check_against_stored, weighting_tables)
print('Project:', ROOT)

Project: C:\Users\ADMIN\OneDrive\Tài liệu\ChatGPT\KLTN\KLTN_clean


## Source lock

In [2]:
checks = verify_sources(ROOT)
print('Frozen source files:', len(checks), '| PASS:', sum(x['status']=='PASS' for x in checks))

Frozen source files: 15 | PASS: 15


## Kiến trúc

Mã C/G/D và head được trích nguyên hàm từ worker M4. Không khởi tạo TensorFlow.

In [3]:
from src.audit import function_source
src = ROOT / 'src/frozen/m4.py'
names = ['tao_classifier', 'tao_generator', 'tao_discriminator', 'tao_head_convex']
methods = function_source(src, names)
for name in names:
    print(f'{name}: {len(methods[name].splitlines())} lines')

tao_classifier: 13 lines
tao_generator: 19 lines
tao_discriminator: 15 lines
tao_head_convex: 10 lines


## Source

Các hàm có thể xem trực tiếp; file gốc được giữ bất biến trong `src/frozen/`.

In [4]:
# Select one function to inspect without constructing a model.
VIEW_FUNCTION = 'tao_generator'
print(methods[VIEW_FUNCTION])

def tao_generator(latent_dim, so_lop):
    L = keras.layers
    noise = L.Input(shape=(latent_dim,))
    labels = L.Input(shape=(so_lop,))
    x = L.Concatenate()([noise, labels])
    x = L.Dense(7*7*256, use_bias=False)(x)
    x = L.BatchNormalization(momentum=0.9)(x)
    x = L.LeakyReLU()(x)
    x = L.Reshape((7,7,256))(x)
    x = L.Conv2DTranspose(128, (5,5), strides=(1,1), padding="same", use_bias=False)(x)
    x = L.BatchNormalization(momentum=0.9)(x)
    x = L.LeakyReLU()(x)
    x = L.Conv2DTranspose(64, (5,5), strides=(2,2), padding="same", use_bias=False)(x)
    x = L.BatchNormalization(momentum=0.9)(x)
    x = L.LeakyReLU()(x)
    x = L.Conv2DTranspose(1, (5,5), strides=(2,2), padding="same", use_bias=False)(x)
    x = L.BatchNormalization(momentum=0.9)(x)
    out = L.Flatten()(x)  # 28×28 = 784
    return keras.Model([noise, labels], out, name="G")


## Xuất xứ

Notebook B gốc vẫn giữ nguyên trong archive để truy xuất quy trình và các baseline classweight/ROS.

In [5]:
original = ROOT / 'archive/legacy/KLTN_3D_HyperGAMO_Protocol_B_v2.ipynb'
print({'protocol_b_notebook': original.is_file(), 'm4_source': src.name})

{'protocol_b_notebook': True, 'm4_source': 'm4.py'}


## Giới hạn

Không thay đổi source huấn luyện đã chạy, không mở test mới và không kết luận PASS khi thiếu output gốc. Kết quả chỉ có giá trị trong phạm vi protocol B của Indian Pines.

In [6]:
import sys
assert 'tensorflow' not in sys.modules, 'Unexpected TensorFlow/model initialization'
print('CPU_ONLY_NO_MODEL_IMPORT_PASS')

CPU_ONLY_NO_MODEL_IMPORT_PASS
